# Zadanie 1

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import plotly.express as px
import plotly.graph_objects as go
import lightgbm as lgb

In [2]:
df = pd.read_csv("pzz_fleet_data_full.csv")
with open("pzz_fleet_physical_limits.json") as f:
    physical_limits = json.load(f)
sensor_cols = ["temperature", "vibration", "speed", "fuel_consumption", "load"]
df_sensors = df[sensor_cols]


In [3]:
def detect_outliers_iqr(df):
    outliers = pd.DataFrame(False, index=df.index, columns=df.columns)
    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers[col] = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)
    return outliers

In [4]:
def detect_outliers_zscore(df):
    outliers = pd.DataFrame(False, index=df.index, columns=df.columns)
    for col in df.columns:
        mean = df[col].mean()
        std = df[col].std()
        zscore = (df[col] - mean) / std
        outliers[col] = np.abs(zscore) > 3
    return outliers

In [5]:
def detect_outliers_iforest(df, contamination=0.03):
    clf = IsolationForest(contamination=contamination, random_state=42)
    clf.fit(df)
    preds = clf.predict(df)
    outliers = (preds == -1)
    return pd.DataFrame(np.tile(outliers[:, np.newaxis], (1, df.shape[1])), columns=df.columns)



In [6]:
iqr_outliers = detect_outliers_iqr(df_sensors)
zscore_outliers = detect_outliers_zscore(df_sensors)
iforest_outliers = detect_outliers_iforest(df_sensors)

In [7]:
print("Liczba outlierów IQR:", iqr_outliers.sum().sum())
print("Liczba outlierów Z-score:", zscore_outliers.sum().sum())
print("Liczba outlierów Isolation Forest:", iforest_outliers.sum().sum())

Liczba outlierów IQR: 1455
Liczba outlierów Z-score: 672
Liczba outlierów Isolation Forest: 1500


In [8]:
overlap = iqr_outliers & zscore_outliers & iforest_outliers
print("Outliery wykryte przez wszystkie 3 metody:", overlap.sum().sum())


Outliery wykryte przez wszystkie 3 metody: 672


In [9]:
def classify_outlier(value, sensor_name):
    limits = physical_limits[sensor_name]  
    if value < limits[0] or value > limits[1]:
        return "sensor_error"
    else:
        return "true_anomaly"


In [10]:
outlier_flags = (iqr_outliers | zscore_outliers | iforest_outliers)

outlier_categories = pd.DataFrame("", index=df_sensors.index, columns=df_sensors.columns)
for col in df_sensors.columns:
    outlier_categories.loc[outlier_flags[col], col] = df_sensors.loc[outlier_flags[col], col].apply(lambda x: classify_outlier(x, col))

print("Błędy sensorów:", (outlier_categories == "sensor_error").sum().sum())
print("Prawdziwe anomalie:", (outlier_categories == "true_anomaly").sum().sum())

Błędy sensorów: 1000
Prawdziwe anomalie: 691


In [11]:
df_removed = df_sensors.mask(outlier_flags)
df_removed = df_removed.dropna()

df_winsor = df_sensors.copy()
for col in df_sensors.columns:
    lower = df_sensors[col].quantile(0.01)
    upper = df_sensors[col].quantile(0.99)
    df_winsor[col] = df_sensors[col].clip(lower, upper)

df_flagged = df_sensors.copy()
for col in df_sensors.columns:
    df_flagged[col+"_flag"] = outlier_categories[col].replace({"": "normal"})




In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

target = pd.Series(0, index=df_sensors.index)  
for col in df_sensors.columns:
    target.loc[outlier_categories[col] == "true_anomaly"] = 1


df_removed_corrected = df_sensors.copy()
for col in df_sensors.columns: 
    df_removed_corrected.loc[outlier_categories[col] == "sensor_error", col] = np.nan
df_removed_corrected = df_removed_corrected.dropna()

X_removed = df_removed_corrected
y_removed = target.loc[X_removed.index]

X_train, X_test, y_train, y_test = train_test_split(X_removed, y_removed, test_size=0.2, random_state=42)

clf_removed = RandomForestClassifier(random_state=42)
clf_removed.fit(X_train, y_train)
y_pred = clf_removed.predict(X_test)
print("F1 score - removed sensor errors only:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


X_winsor = df_winsor
y_winsor = target.loc[X_winsor.index]

X_train, X_test, y_train, y_test = train_test_split(X_winsor, y_winsor, test_size=0.2, random_state=42)

clf_winsor = RandomForestClassifier(random_state=42)
clf_winsor.fit(X_train, y_train)
y_pred = clf_winsor.predict(X_test)
print("F1 score - winsorized:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


X_flagged = df_flagged.drop(columns=[col+"_flag" for col in sensor_cols])
y_flagged = target.loc[X_flagged.index]

X_train, X_test, y_train, y_test = train_test_split(X_flagged, y_flagged, test_size=0.2, random_state=42)

clf_flagged = RandomForestClassifier(random_state=42)
clf_flagged.fit(X_train, y_train)
y_pred = clf_flagged.predict(X_test)
print("F1 score - flagged anomalies:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


F1 score - removed sensor errors only: 0.9928057553956835
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1890
           1       1.00      0.99      0.99        70

    accuracy                           1.00      1960
   macro avg       1.00      0.99      1.00      1960
weighted avg       1.00      1.00      1.00      1960

F1 score - winsorized: 0.9836065573770492
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1940
           1       0.97      1.00      0.98        60

    accuracy                           1.00      2000
   macro avg       0.98      1.00      0.99      2000
weighted avg       1.00      1.00      1.00      2000

F1 score - flagged anomalies: 0.9836065573770492
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1940
           1       0.97      1.00      0.98        60

    accuracy                     

In [13]:
sensor = df_sensors.columns[0]
fig = go.Figure()
fig.add_trace(go.Box(y=df_sensors[sensor], name="Sensor readings", boxpoints="all", jitter=0.3, pointpos=-1.8))
fig.update_layout(title=f"Outliery dla sensora {sensor}", yaxis_title="Wartość")
fig.show()

In [14]:
def pipeline(row):
    decisions = {}
    for col in df_sensors.columns:
        val = row[col]
        limits = physical_limits[col]  # [min, max]
        
        if val < limits[0] or val > limits[1]:
            decisions[col] = "remove"
        elif row[col+"_flag"] == "true_anomaly":  # sprawdzamy flagę
            decisions[col] = "alert"
        else:
            decisions[col] = "keep"
    return decisions


In [15]:
example_decision = pipeline(df_flagged.iloc[0])
print("Decyzje pipeline dla pierwszego wiersza:", example_decision)

Decyzje pipeline dla pierwszego wiersza: {'temperature': 'keep', 'vibration': 'keep', 'speed': 'keep', 'fuel_consumption': 'keep', 'load': 'keep'}


# Zadanie 2

import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import plotly.express as px


In [16]:

sales = pd.read_csv("wroclaw_retail.csv", parse_dates=["date"])
products = pd.read_csv("wroretail_products.csv")
stores = pd.read_csv("wroretail_stores.csv")
holidays = pd.read_csv("polish_holidays_2021_2024.csv", parse_dates=["date"])


sales = sales.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)


In [17]:

df = sales.merge(products, on="product_id", how="left")


df = df.merge(stores, on="store_id", how="left")


df = df.merge(
    holidays,
    on="date",
    how="left"
)


df["is_public_holiday"] = df["is_public_holiday"].fillna(0)
df["is_shopping_boost"] = df["is_shopping_boost"].fillna(0)
df["category_boost"] = df["category_boost"].fillna(0)
df["holiday_name"] = df["holiday_name"].fillna("none")

df.head()


,date,store_id,product_id,sales,product_name,category,min_price,max_price,base_popularity,store_name,city,store_type,opening_year,sales_multiplier,holiday_name,is_public_holiday,is_shopping_boost,category_boost
0,2022-01-01,1,1,712.05,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,Nowy Rok,1.0,0.0,0
1,2022-01-02,1,1,798.06,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
2,2022-01-03,1,1,482.66,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
3,2022-01-04,1,1,483.14,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0
4,2022-01-05,1,1,619.95,Telewizor Samsung 55,Elektronika,1200,2500,1.2,WroRetail Wroclaw Centrum,Wroclaw,hipermarket,2015,1.5,none,0.0,0.0,0


In [18]:
df["day_of_week"] = df["date"].dt.weekday
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year

df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


In [19]:
def season_pl(month):
    if month in [12, 1, 2]:
        return "zima"
    if month in [3, 4, 5]:
        return "wiosna"
    if month in [6, 7, 8]:
        return "lato"
    return "jesien"

df["season"] = df["month"].apply(season_pl).astype("category")


In [20]:
dow = df.groupby("day_of_week")["sales"].mean().reset_index()

px.bar(
    dow,
    x="day_of_week",
    y="sales",
    title="Średnia sprzedaż vs dzień tygodnia"
).show()


In [21]:
month = df.groupby("month")["sales"].mean().reset_index()

px.line(
    month,
    x="month",
    y="sales",
    title="Średnia sprzedaż vs miesiąc"
).show()


In [22]:
for lag in [1, 7, 28]:
    df[f"sales_lag_{lag}"] = (
        df
        .groupby(["store_id", "product_id"])["sales"]
        .shift(lag)
    )


In [23]:
df["rolling_mean_7"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

df["rolling_mean_28"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(28)
    .mean()
)

df["rolling_std_7"] = (
    df.groupby(["store_id", "product_id"])["sales"]
    .shift(1)
    .rolling(7)
    .std()
)


In [24]:
df["avg_sales_product"] = (
    df.groupby("product_id")["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [25]:
df["avg_sales_store"] = (
    df.groupby("store_id")["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [26]:
df["avg_sales_category_dow"] = (
    df.groupby(["category", "day_of_week"])["sales"]
    .shift(1)
    .expanding()
    .mean()
)


In [27]:
df["product_percentile_in_category"] = (
    df.groupby(["category", "date"])["sales"]
    .shift(1)
    .rank(pct=True)
)


In [28]:
df_model = df.dropna().copy()

target = "sales"

features = [
    "day_of_week", "month", "quarter", "year", "is_weekend",
    "is_public_holiday", "is_shopping_boost", "category_boost",
    "base_popularity", "sales_multiplier",
    "sales_lag_1", "sales_lag_7", "sales_lag_28",
    "rolling_mean_7", "rolling_mean_28", "rolling_std_7",
    "avg_sales_product", "avg_sales_store",
    "avg_sales_category_dow",
    "product_percentile_in_category"
]

df_model = pd.get_dummies(
    df_model,
    columns=["season", "store_type", "category"],
    drop_first=True
)

X = df_model[features + [c for c in df_model.columns if c.startswith(("season_", "store_type_", "category_"))]]
y = df_model[target]


In [29]:
split_idx = int(len(df_model) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]


In [30]:
baseline_features = ["day_of_week", "month", "is_weekend"]

Xb_train = X_train[baseline_features]
Xb_test = X_test[baseline_features]

baseline = lgb.LGBMRegressor(random_state=42)
baseline.fit(Xb_train, y_train)

baseline_pred = baseline.predict(Xb_test)
baseline_rmse = mean_squared_error(y_test, baseline_pred)
baseline_rmse


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000240 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 22
[LightGBM] [Info] Number of data points in the train set: 26956, number of used features: 3
[LightGBM] [Info] Start training from score 371.125031


46122.42597693682

In [31]:

print(X_train.dtypes)

for c in X_train.columns:
    if pd.api.types.is_object_dtype(X_train[c]):
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce").fillna(0)
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(0)




print(X_train.dtypes)
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]
X_train["category_boost"] = pd.to_numeric(X_train["category_boost"], errors="coerce").fillna(0)
X_test["category_boost"] = pd.to_numeric(X_test["category_boost"], errors="coerce").fillna(0)




day_of_week                         int32
month                               int32
quarter                             int32
year                                int32
is_weekend                          int64
is_public_holiday                 float64
is_shopping_boost                 float64
category_boost                     object
base_popularity                   float64
sales_multiplier                  float64
sales_lag_1                       float64
sales_lag_7                       float64
sales_lag_28                      float64
rolling_mean_7                    float64
rolling_mean_28                   float64
rolling_std_7                     float64
avg_sales_product                 float64
avg_sales_store                   float64
avg_sales_category_dow            float64
product_percentile_in_category    float64
category_boost                     object
season_lato                          bool
season_wiosna                        bool
season_zima                       

In [32]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = mean_squared_error(y_test, pred)
rmse


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000643 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2608
[LightGBM] [Info] Number of data points in the train set: 26956, number of used features: 24
[LightGBM] [Info] Start training from score 371.125031


3003.7635063914554

In [33]:
importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

import plotly.express as px

fig = px.bar(
    importance.head(20),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importance"
)
fig.show()

